In [1]:
import pandas as pd
import simpful as sf

# ==================================================
# 1. CONFIGURAÇÃO DO SISTEMA FUZZY (Módulos E2 e E3)
# ==================================================
FS = sf.FuzzySystem(show_status=False)

# VARIÁVEL 1: Temperatura da Massa (E2/E3 - Escoamento)
# Faixa de operação: 1300°C a 1600°C
t_baixa = sf.FuzzySet(function=sf.Trapezoidal_MF(a=1300, b=1300, c=1350, d=1400), term="viscoso")
t_ideal = sf.FuzzySet(function=sf.Triangular_MF(a=1380, b=1425, c=1470), term="ideal")
t_alta  = sf.FuzzySet(function=sf.Trapezoidal_MF(a=1430, b=1450, c=1600, d=1600), term="fluido")
FS.add_linguistic_variable("Temp_Vidro", sf.LinguisticVariable([t_baixa, t_ideal, t_alta], universe_of_discourse=[1300, 1600]))

#VARIÁVEL 2: Tendência (Variação no Tempo - dt)
# Mede a velocidade de mudança térmica (°C/min) vinda da planilha
v_esf = sf.FuzzySet(function=sf.Triangular_MF(a=-10, b=-10, c=0), term="esfriando")
v_est = sf.FuzzySet(function=sf.Triangular_MF(a=-2, b=0, c=2), term="estavel")
v_aqu = sf.FuzzySet(function=sf.Triangular_MF(a=0, b=10, c=10), term="aquecendo")
FS.add_linguistic_variable("Tendencia", sf.LinguisticVariable([v_esf, v_est, v_aqu], universe_of_discourse=[-10, 10]))

# VARIÁVEL 3: Temperatura do Sensor (Segurança de Hardware)
# Proteção para o sensor de nível não queimar (< 80°C)
s_ok = sf.FuzzySet(function=sf.Trapezoidal_MF(a=0, b=0, c=50, d=70), term="seguro")
s_pr = sf.FuzzySet(function=sf.Trapezoidal_MF(a=65, b=75, c=80, d=80), term="perigo")
FS.add_linguistic_variable("Temp_Sensor", sf.LinguisticVariable([s_ok, s_pr], universe_of_discourse=[0, 80]))

#  VARIÁVEL DE SAÍDA: Potência de Resfriamento (Atuador)
# Geometria trapezoidal para resposta decisiva (86%+)
out_b = sf.FuzzySet(function=sf.Trapezoidal_MF(a=0, b=0, c=10, d=30), term="baixo")
out_m = sf.FuzzySet(function=sf.Trapezoidal_MF(a=60, b=95, c=100, d=100), term="maximo")
FS.add_linguistic_variable("Resfriamento", sf.LinguisticVariable([out_b, out_m], universe_of_discourse=[0, 100]))

# =================================================================
# 2. BASE DE REGRAS (Lógica de Mamdani)
# =================================================================
FS.add_rules([
    "IF (Temp_Sensor IS perigo) THEN (Resfriamento IS maximo)",
    "IF (Temp_Vidro IS fluido) AND (Tendencia IS aquecendo) AND (Temp_Sensor IS seguro) THEN (Resfriamento IS maximo)",
    "IF (Temp_Vidro IS ideal) AND (Tendencia IS estavel) AND (Temp_Sensor IS seguro) THEN (Resfriamento IS baixo)",
    "IF (Tendencia IS esfriando) AND (Temp_Sensor IS seguro) THEN (Resfriamento IS baixo)"
])

# =================================================================
# 3. INTEGRAÇÃO COM PLANILHAS (E1, E2, E3, E4)
# =================================================================
def pipeline_producao_vidro(caminho_excel):
    # Carregamento dos dados coletados
    df = pd.read_excel(caminho_excel)
    
    # Cálculo automático da tendência (Derivada dT/dt) se não estiver na planilha
    if 'Tendencia' not in df.columns:
        df['Tendencia'] = df['Temp_Vidro'].diff().fillna(0)

    decisoes_ia = []
    custos_estimados = []

    # Processamento linear linha por linha (instantes de tempo)
    for i, linha in df.iterrows():
        # -- Execução do Módulo E3 (Fuzzy) --
        FS.set_variable("Temp_Vidro", linha['Temp_Vidro'])
        FS.set_variable("Tendencia", linha['Tendencia'])
        FS.set_variable("Temp_Sensor", linha['Temp_Sensor'])
        
        resultado_fuzzy = FS.inference()['Resfriamento']
        decisoes_ia.append(resultado_fuzzy)
        
        # -- Execução do Módulo E4 (Estimativa de Custos) --
        # Exemplo de integração financeira conforme E4
        custo = (linha['Consumo_kWh'] * 0.95) + linha['Custo_Materia_Prima_E1']
        custos_estimados.append(custo)

    # Inserindo resultados no DataFrame final
    df['Sugestao_IA_Resfriamento'] = decisoes_ia
    df['E4_Custo_Total'] = custos_estimados
    
    return df

# Exemplo para teste imediato:
# df_final = pipeline_producao_vidro("sua_planilha_de_coleta.xlsx")
# print(df_final.head())

TypeError: FuzzySystem.__init__() got an unexpected keyword argument 'show_status'